## Imports/dependencies

In [ ]:
import os
from pathlib import Path

!pip install -q einops omegaconf pytorch-lightning==1.9.5 transformers open_clip_torch gradio
!pip install -q opencv-python pillow matplotlib

import torch, cv2, pytorch_lightning
print(torch.__version__)
print(torch.cuda.is_available())

## Mount to Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# All project files live here — change this if you want a different folder name
PROJECT_DIR = Path("/content/drive/MyDrive/controlnet_project")

# Create all needed directories
CODE_DIR = PROJECT_DIR / "code"
SPLIT_DIR = PROJECT_DIR / "splits"
CKPT_DIR = PROJECT_DIR / "checkpoints"

for d in [CODE_DIR, SPLIT_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Project directory: {PROJECT_DIR}')
print('Directories ready')

## Clone repo from git; download dataset and models

In [ ]:
!git clone https://github.com/lllyasviel/ControlNet.git
%cd ControlNet

In [ ]:
%cd /content/ControlNet
!mkdir -p training
!wget -O training/fill50k.zip https://huggingface.co/lllyasviel/ControlNet/resolve/main/training/fill50k.zip
!unzip -q training/fill50k.zip -d training/

In [ ]:
%cd /content/ControlNet
!python tutorial_dataset_test.py

In [ ]:
%cd /content/ControlNet
!mkdir -p models
!wget -O models/v1-5-pruned.ckpt https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned.ckpt

In [ ]:
%cd /content/ControlNet
!pwd
!ls -lh models

In [ ]:
%cd /content/ControlNet

# Patch torch.load in tool_add_control.py to work with PyTorch 2.6+
!python - <<'PY'
from pathlib import Path

p = Path("tool_add_control.py")
text = p.read_text()

old = "pretrained_weights = torch.load(input_path)"
new = "pretrained_weights = torch.load(input_path, map_location='cpu', weights_only=False)"

if old in text:
    text = text.replace(old, new)
    p.write_text(text)
    print("Patched tool_add_control.py")
else:
    print("Could not find exact line. Showing torch.load lines:")
    for i, line in enumerate(text.splitlines(), 1):
        if "torch.load" in line:
            print(i, line)

## Create initial, untrained Controlnet

In [ ]:
!python tool_add_control.py ./models/v1-5-pruned.ckpt ./models/control_sd15_ini.ckpt

In [ ]:
!ls -lh models | grep control_sd15_ini

## Patch some files

In [ ]:
%cd /content/ControlNet

!python - <<'PY'
from pathlib import Path

# Find where on_train_batch_start is defined
for path in Path(".").rglob("*.py"):
    text = path.read_text(errors="ignore")
    if "def on_train_batch_start" in text:
        print(path)
        for i, line in enumerate(text.splitlines(), 1):
            if "def on_train_batch_start" in line:
                print(i, line)

In [ ]:
%cd /content/ControlNet

from pathlib import Path
import re

patched = False
for path in Path(".").rglob("*.py"):
    text = path.read_text(errors="ignore")
    new_text = re.sub(
        r"def on_train_batch_start\(self,\s*batch,\s*batch_idx,\s*dataloader_idx\):",
        "def on_train_batch_start(self, batch, batch_idx, dataloader_idx=0):",
        text
    )
    if new_text != text:
        path.write_text(new_text)
        print("Patched:", path)
        patched = True

if not patched:
    print("No exact signature found. Showing any on_train_batch_start definitions:")
    for path in Path(".").rglob("*.py"):
        text = path.read_text(errors="ignore")
        if "def on_train_batch_start" in text:
            for i, line in enumerate(text.splitlines(), 1):
                if "def on_train_batch_start" in line:
                    print(path, i, line)

In [ ]:
%cd /content/ControlNet

from pathlib import Path
import re

for path in Path(".").rglob("*.py"):
    text = path.read_text(errors="ignore")
    if "def on_train_batch_end" in text:
        print("Found in", path)
        for i, line in enumerate(text.splitlines(), 1):
            if "def on_train_batch_end" in line:
                print(i, line)

# Patch callback signature
patched = False
for path in Path(".").rglob("*.py"):
    text = path.read_text(errors="ignore")
    new_text = re.sub(
        r"def on_train_batch_end\(self,\s*trainer,\s*pl_module,\s*outputs,\s*batch,\s*batch_idx,\s*dataloader_idx\):",
        "def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx=0):",
        text
    )
    if new_text != text:
        path.write_text(new_text)
        print("Patched:", path)
        patched = True

if not patched:
    print("No exact signature patched. Check printed definitions above.")

## Dataset utils

In [ ]:
from torch.utils.data import DataLoader, random_split, Subset

def split_dataset(dataset, load_existing_split=True, batch_size=8):
  SPLIT_PATH = f"{SPLIT_DIR}/split_seed42.pt"
  if load_existing_split == True:
    print(f"Reusing saved splits in {SPLIT_PATH}.")
    split = torch.load(SPLIT_PATH)

    # Recreate subsets
    train_dataset = Subset(dataset, split["train"])
    val_dataset   = Subset(dataset, split["val"])
    test_dataset  = Subset(dataset, split["test"])
  else:
    print("Generating a new dataset split.")
    TRAIN_SPLIT = 0.8
    VAL_SPLIT = 0.1
    n = len(dataset)

    n_train = int(TRAIN_SPLIT * n)
    n_val = int(VAL_SPLIT * n)
    n_test = n - n_train - n_val

    generator = torch.Generator().manual_seed(42)

    train_dataset, val_dataset, test_dataset = random_split(
        dataset,
        [n_train, n_val, n_test],
        generator=generator,
    )

    # Persist split indices to Drive
    os.makedirs(os.path.dirname(SPLIT_PATH), exist_ok=True)

    split = {
        "train": train_dataset.indices,
        "val": val_dataset.indices,
        "test": test_dataset.indices,
    }

    torch.save(split, SPLIT_PATH)
    print(f"Saved new split to {SPLIT_PATH}")

  # Create dataloaders
  print(f"Creating dataloaders with batch size {batch_size}.")
  train_dataloader = DataLoader(
      train_dataset,
      num_workers=8,
      batch_size=batch_size,
      shuffle=True, # only shuffle in the train dataloader
  )

  val_dataloader = DataLoader(
      val_dataset,
      num_workers=8,
      batch_size=batch_size,
      shuffle=False,
  )

  test_dataloader = DataLoader(
      test_dataset,
      num_workers=8,
      batch_size=batch_size,
      shuffle=False,
  )
  return train_dataloader, val_dataloader, test_dataloader

## Setup and Train ControlNet Model

This block initializes the ControlNet model, loads pre-trained weights, configures training parameters like learning rate and batch size, prepares data loaders, and sets up callbacks for logging and checkpointing. It then starts the training process using PyTorch Lightning.

In [ ]:
# !python tutorial_train.py
from share import *

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from torch.utils.data import DataLoader
from tutorial_dataset import MyDataset
from cldm.logger import ImageLogger
from cldm.model import create_model, load_state_dict
import os
import pathlib


# Configs
resume_path = './models/control_sd15_ini.ckpt'
batch_size = 4
logger_freq = 300
learning_rate = 1e-5
sd_locked = True
only_mid_control = False


# First use cpu to load models. Pytorch Lightning will automatically move it to GPUs.
model = create_model('./models/cldm_v15.yaml').cpu()
model.load_state_dict(load_state_dict(resume_path, location='cpu'))
model.learning_rate = learning_rate
model.sd_locked = sd_locked
model.only_mid_control = only_mid_control


# Misc
dataset = MyDataset()
train_dataloader, val_dataloader, test_dataloader = split_dataset(dataset)

# Callbacks
checkpoint_callback = ModelCheckpoint(
    # dirpath = './checkpoints/',
    dirpath = CKPT_DIR,
    every_n_train_steps=2000,
    save_top_k = -1,
    save_last = True,
    filename = 'controlnet-{epoch:02d}-{step:06d}'
)
# dataloader = DataLoader(dataset, num_workers=0, batch_size=batch_size, shuffle=True)
logger = ImageLogger(batch_frequency=logger_freq)
trainer = pl.Trainer(gpus=1, precision=32, callbacks=[logger, checkpoint_callback], max_epochs=2)


# Train!
torch.serialization.add_safe_globals([pathlib.PosixPath]) # Allow Lightning checkpoint objects under PyTorch 2.6+ weights_only behavior
ckpt_path = f"{CKPT_DIR}/controlnet-epoch=00-step=004000.ckpt"
trainer.fit(
    model,
    train_dataloader,
    val_dataloader,
    ckpt_path=ckpt_path
)
# trainer.fit(model, train_dataloader, val_dataloader)

# print where checkpoins were saved
ckpts = sorted(os.listdir(CKPT_DIR))
print("Saved checkpoints:")
for c in ckpts:
  print(" ", c)

In [ ]:
trainer.save_checkpoint(f"{CKPT_DIR}/controlnet-final.ckpt")

In [ ]:
%cd /content/ControlNet
!find . -type f -name "*.ckpt" | sort

In [ ]:
!find . -type f \( -name "*.png" -o -name "*.jpg" \) | head -100

In [ ]:
!ls -R lightning_logs | head -100

In [ ]:
%cd /content/ControlNet
!find ./image_log ./lightning_logs -type f \( -name "*.png" -o -name "*.jpg" \) | sort | tail -30

## Visualize Training Progress

This cell displays the latest reconstruction and sample images generated during training, giving you a visual insight into the model's learning process.

In [ ]:
from IPython.display import Image, display
import glob

recons = sorted(glob.glob("/content/ControlNet/image_log/train/reconstruction*.png"))
samples = sorted(glob.glob("/content/ControlNet/image_log/train/samples*.png"))

print("Latest reconstruction:", recons[-1])
display(Image(filename=recons[-1]))

print("Latest sample:", samples[-1])
display(Image(filename=samples[-1]))

## Model Inference and Visualizations

The following cells use the trained ControlNet model to generate new images based on a given control condition and prompt from the dataset. It shows the control condition, the target image, and the generated image for comparison.

In [ ]:
%cd /content/ControlNet

from tutorial_dataset import MyDataset
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

dataset = MyDataset()

idx = 5
item = dataset[idx]

print("Prompt:", item["txt"])

hint = item["hint"] # condition image, range [0,1]
jpg = item["jpg"] # target image, range [-1,1]

hint_img = (hint * 255).astype(np.uint8)
target_img = ((jpg + 1) * 127.5).clip(0, 255).astype(np.uint8)

plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.imshow(hint_img)
plt.title("Control condition / hint")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(target_img)
plt.title("Target image")
plt.axis("off")

plt.show()

In [ ]:
%cd /content/ControlNet

import torch
import einops
import numpy as np
import matplotlib.pyplot as plt

from tutorial_dataset import MyDataset
from cldm.model import create_model, load_state_dict
from cldm.ddim_hacked import DDIMSampler

# Load trained model
ckpt_path = "./lightning_logs/version_3/checkpoints/epoch=0-step=500.ckpt"

model = create_model("./models/cldm_v15.yaml").cpu()
model.load_state_dict(load_state_dict(ckpt_path, location="cpu"), strict=False)
model = model.cuda()
model.eval()

sampler = DDIMSampler(model)

# Pick one dataset example
dataset = MyDataset()
idx = 0
item = dataset[idx]

prompt = item["txt"]
condition = item["hint"]   # [0,1], HWC
target = item["jpg"]       # [-1,1], HWC

H, W, C = condition.shape

# Control tensor: HWC [0,1] -> BCHW cuda
control = torch.from_numpy(condition.copy()).float().cuda()
control = control.unsqueeze(0)  # BHWC
control = einops.rearrange(control, "b h w c -> b c h w").clone()

# Conditioning
cond = {
    "c_concat": [control],
    "c_crossattn": [model.get_learned_conditioning([prompt])]
}
un_cond = {
    "c_concat": [control],
    "c_crossattn": [model.get_learned_conditioning([""])]
}

# Generate
ddim_steps = 20
scale = 9.0
shape = (4, H // 8, W // 8)

with torch.no_grad():
    samples, _ = sampler.sample(
        S=ddim_steps,
        conditioning=cond,
        batch_size=1,
        shape=shape,
        verbose=False,
        unconditional_guidance_scale=scale,
        unconditional_conditioning=un_cond,
        eta=0.0,
    )

    x_samples = model.decode_first_stage(samples)
    x_samples = (einops.rearrange(x_samples, "b c h w -> b h w c") * 127.5 + 127.5)
    x_samples = x_samples.cpu().numpy().clip(0, 255).astype(np.uint8)

generated_img = x_samples[0]
condition_img = (condition * 255).astype(np.uint8)
target_img = ((target + 1.0) * 127.5).clip(0, 255).astype(np.uint8)

# Display
print("Prompt:", prompt)

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(condition_img)
plt.title("Control condition")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(target_img)
plt.title("Target image")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(generated_img)
plt.title("Generated image")
plt.axis("off")

plt.show()

In [ ]:
%cd /content/ControlNet

import torch
import einops
import numpy as np
import matplotlib.pyplot as plt

from tutorial_dataset import MyDataset
from cldm.model import create_model, load_state_dict
from cldm.ddim_hacked import DDIMSampler

# Load trained checkpoint
ckpt_path = "./lightning_logs/version_3/checkpoints/epoch=0-step=500.ckpt"

model = create_model("./models/cldm_v15.yaml").cpu()
model.load_state_dict(load_state_dict(ckpt_path, location="cpu"), strict=False)
model = model.cuda()
model.eval()

sampler = DDIMSampler(model)

# Pick example
dataset = MyDataset()
idx = 45000   # use a held-out-ish example, or change to 0 for training-ish example
item = dataset[idx]

prompt = item["txt"]
condition = item["hint"]   # HWC, [0,1]
target = item["jpg"]       # HWC, [-1,1]

H, W, C = condition.shape

# Prepare condition tensor
control = torch.from_numpy(condition.copy()).float().cuda()
control = control.unsqueeze(0)
control = einops.rearrange(control, "b h w c -> b c h w").clone()

cond = {
    "c_concat": [control],
    "c_crossattn": [model.get_learned_conditioning([prompt])]
}
un_cond = {
    "c_concat": [control],
    "c_crossattn": [model.get_learned_conditioning([""])]
}

# Generate
ddim_steps = 20
scale = 9.0
shape = (4, H // 8, W // 8)

with torch.no_grad():
    samples, _ = sampler.sample(
        S=ddim_steps,
        conditioning=cond,
        batch_size=1,
        shape=shape,
        verbose=False,
        unconditional_guidance_scale=scale,
        unconditional_conditioning=un_cond,
        eta=0.0,
    )

    x_samples = model.decode_first_stage(samples)
    x_samples = (einops.rearrange(x_samples, "b c h w -> b h w c") * 127.5 + 127.5)
    generated = x_samples.cpu().numpy().clip(0, 255).astype(np.uint8)[0]

condition_img = (condition * 255).astype(np.uint8)
target_img = ((target + 1.0) * 127.5).clip(0, 255).astype(np.uint8)

# Display
print("Prompt:", prompt)

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(condition_img)
plt.title("Control")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(generated)
plt.title("Prediction")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(target_img)
plt.title("Ground Truth")
plt.axis("off")

plt.show()